In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from PIL import Image
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

In [2]:
os.chdir("/home/selc-a4-sr2/Solar_Rooftop_Detection")

In [3]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [4]:
class SegmentationDataset(Dataset):
    def __init__(self, root, image_folder="images", mask_folder="masks", transforms=None):
        self.root = root
        self.transforms = transforms
        self.image_folder = os.path.join(root, image_folder)
        self.mask_folder = os.path.join(root, mask_folder)
        self.image_names = sorted(os.listdir(self.image_folder))
        self.mask_names = sorted(os.listdir(self.mask_folder))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_folder, self.image_names[idx])
        mask_path = os.path.join(self.mask_folder, self.mask_names[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale mask
        if self.transforms:
            image = self.transforms(image)
            mask = transforms.ToTensor()(mask)  # Mask to tensor (0-1 range)
        return {"image": image, "mask": mask}

In [5]:
def iou_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-10) if union > 0 else 1.0

def f1_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    tp = np.sum(y_true * y_pred)
    fp = np.sum(y_pred) - tp
    fn = np.sum(y_true) - tp
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    return 2 * (precision * recall) / (precision + recall + 1e-10)

In [6]:
import pandas as pd

In [7]:
def evaluate_model(model, dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set to evaluation mode

    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()  # Same loss as training
    num_samples = 0
    all_matrics = []

    with torch.no_grad():  # No gradient computation
        for sample in tqdm(dataloader):
            inputs = sample["image"].to(device)
            masks = sample["mask"].to(device)
            outputs = model(inputs)["out"]  # Shape: (batch, 1, H, W)
            loss = criterion(outputs, masks)
            total_loss += loss.item() * inputs.size(0)

            # Convert logits to binary predictions
            
            preds = torch.sigmoid(outputs) > 0.5  # Threshold at 0.5
            preds_ = preds
            masks_ = masks
            preds = preds.cpu().numpy().astype(np.uint8)
            masks = masks.cpu().numpy().astype(np.uint8)

            # Compute metrics per batch
            for i in range(inputs.size(0)):
                all_matrics.append(compute_metrics(preds_[i], masks_[i]))
                # Compute IoU and F1 score
                total_iou += iou_score(masks[i], preds[i])
                total_f1 += f1_score(masks[i], preds[i])
            num_samples += inputs.size(0)

    avg_loss = total_loss / num_samples
    avg_iou = total_iou / num_samples
    avg_f1 = total_f1 / num_samples
    
    df = pd.DataFrame(all_matrics, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall","region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
    df.to_csv("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/deeplabv3/metrics.csv", index=False)
    return avg_loss, avg_iou, avg_f1, all_matrics

In [8]:
data_dir = "/home/selc-a4-sr2/Solar_Rooftop_Detection/Arial_validation_images"  # Replace with your data directory
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
])

In [9]:
dataset = SegmentationDataset(data_dir, transforms=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [10]:
model = torch.load("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/deeplabv3/deeplab_rooftop_full_50.pth")
print("Model loaded successfully")

/tmp/ipykernel_2494803/2166194875.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/deeplabv3/

Model loaded successfully


In [11]:
avg_loss, avg_iou, avg_f1, matrixs_array = evaluate_model(model, dataloader)

  5%|▍         | 7/144 [00:09<03:09,  1.38s/it]/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 due to no true or predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
  6%|▌         | 8/144 [00:11<03:06,  1.37s/it]/home/selc-a4-sr2/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 due to no true or predicted samples. Use `zero_division` parameter to control this beha

In [12]:
matrixs_array

[[0.7324267469047758,
  0.8455500334583951,
  0.9608192443847656,
  0.9432422458188651,
  0.7661947457349976,
  0.7324267469047758,
  0.8455500334583951,
  0.9432422458188651,
  0.7661947457349976,
  1.0],
 [0.8383617542381014,
  0.9120748430556375,
  0.9630546569824219,
  0.9478816297840824,
  0.878874828866737,
  0.8383617542381014,
  0.9120748430556375,
  0.9478816297840824,
  0.878874828866737,
  1.0],
 [0.6129564115098138,
  0.7600408878204634,
  0.8663473129272461,
  0.8620032080535349,
  0.6796484566389025,
  0.6129564115098138,
  0.7600408878204634,
  0.8620032080535349,
  0.6796484566389025,
  1.0],
 [0.7656458013718603,
  0.8672699822093125,
  0.9294900894165039,
  0.9155027800623856,
  0.823865070432143,
  0.7656458013718603,
  0.8672699822093125,
  0.9155027800623856,
  0.823865070432143,
  1.0],
 [0.8889615051823487,
  0.9412171743505529,
  0.9539833068847656,
  0.9260990683870102,
  0.956837062760387,
  0.8889615051823487,
  0.9412171743505529,
  0.9260990683870102,
  0.9

In [13]:
import pandas as pd
metrics_df = pd.read_csv("/home/selc-a4-sr2/Solar_Rooftop_Detection/BaseLineModels/deeplabv3/metrics.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.648217
pixel_dice                 0.746504
pixel_accuracy             0.959552
pixel_precision            0.883239
pixel_recall               0.790907
region_iou                 0.717661
region_dice                0.815948
region_precision           0.883239
region_recall              0.790907
region_success_accuracy    0.911458
dtype: float64

In [14]:
avg_loss, avg_iou, avg_f1

(0.12398453354479796, 0.7141777280804783, 0.7444116576813686)